In [1]:
import os
import shutil
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_curve, auc, classification_report, balanced_accuracy_score, precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando em: {device}")


DATA_DIR = "../dataset/preprocess_imgs"
MODELS_DIR = "../models"
RUNS_DIR = "../runs/FusNet_run"
MODEL_SAVE_PATH = os.path.join(MODELS_DIR, "multi_fusnet_best.pth")
TEMP_MODEL_PATH = os.path.join(MODELS_DIR, "temp_model.pth")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

IMAGE_SIZE = (512, 512)
BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 0.001
PATIENCE = 10

Treinando em: cuda


### Funções do Modelo

In [2]:
# 1. Huberloss
class ArticleHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, pred_logits, target):
        pred_probs = torch.softmax(pred_logits, dim=-1)
        target_one_hot = F.one_hot(target, num_classes=pred_logits.size(-1)).float()
        
        # error1 = variação entre previsto e original
        error1 = torch.abs(pred_probs - target_one_hot)
        
        # Equações
        quadratic = 0.5 * (error1 ** 2)
        linear = self.delta * error1 - 0.5 * (self.delta ** 2)
        
        loss = torch.where(error1 <= self.delta, quadratic, linear)
        
        return loss.mean()

# 2. Componentes da Multi-FusNet
class ResidualChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResidualGroup(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.rca = ResidualChannelAttention(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = x
        out = self.relu(self.conv1(x))
        out = self.rca(out)
        out = self.conv2(out)
        return out + res

class MultiFusNetCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.rg_path1_1 = ResidualGroup(64)
        self.rg_path1_2 = ResidualGroup(64)
        self.rg_path2 = ResidualGroup(64)

        self.ps_high = nn.Sequential(
            nn.Conv2d(64, 64 * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2)
        )
        self.ps_low = nn.Sequential(
            nn.Conv2d(16, 64 * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2)
        )
        self.fusion_conv = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        low_level = self.conv1(x)  
        x_mid = self.conv2(low_level)
        high_level = self.conv3(x_mid) 

        path1 = self.rg_path1_2(self.rg_path1_1(high_level))
        path2 = self.rg_path2(high_level)
        multi_out = path1 + path2

        multi_up = self.ps_high(multi_out) 
        low_pooled = F.adaptive_max_pool2d(low_level, multi_out.shape[2:]) 
        low_up = self.ps_low(low_pooled)
        
        fused = self.fusion_conv(multi_up + low_up)

        out = self.global_pool(fused)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        
        return out

### Divisão do Dataset (70/10/20)

In [3]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
num_classes = len(full_dataset.classes)

print(f"Classes encontradas: {full_dataset.classes}")
print(f"Total de imagens no dataset: {len(full_dataset)}")

targets = full_dataset.targets
indices = list(range(len(full_dataset)))

# 1. Separamos 20% para o Teste final e 80% para uma base Temporária (Treino + Validação)
temp_idx, test_idx, temp_targets, test_targets = train_test_split(
    indices, targets, 
    test_size=0.20, 
    stratify=targets, 
    random_state=42
)

# 2. Dos 80% restantes, separamos 12.5% (que equivale a exatos 10% do total geral) para Validação
train_idx, val_idx = train_test_split(
    temp_idx, 
    test_size=0.125, 
    stratify=temp_targets, 
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n[DIVISÃO CONCLUÍDA]")
print(f"Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_dataset)}")

Classes encontradas: ['healthy', 'red_spider_mite', 'rust']
Total de imagens no dataset: 1426

[DIVISÃO CONCLUÍDA]
Treino: 997 | Validação: 143 | Teste: 286


### Treinamento

In [4]:
# Instancia o modelo
model = MultiFusNetCNN(num_classes=num_classes).to(device)

In [5]:
# Article huberloss equation
criterion = ArticleHuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [6]:
# Instancia o modelo
model = MultiFusNetCNN(num_classes=num_classes).to(device)

# Métricas de controle (incluindo train_acc)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
epochs_no_improve = 0  

train_size = len(train_loader.dataset)
val_size = len(val_loader.dataset)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    train_correct = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        # Contabiliza acertos no treinamento
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)
        
    epoch_train_loss = running_loss / train_size
    epoch_train_acc = train_correct.double() / train_size
    
    # Validação
    model.eval()
    val_loss = 0.0
    val_correct = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)

    epoch_val_loss = val_loss / val_size
    epoch_val_acc = val_correct.double() / val_size
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc.item())
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc.item())

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}"
    )

    # EarlyStop
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), TEMP_MODEL_PATH)
        print(f"  -> Melhoria detectada! Loss caiu para {best_val_loss:.4f}. (Patience: 0/{PATIENCE})")
    else:
        epochs_no_improve += 1
        print(f"  -> Sem melhoria. (Patience: {epochs_no_improve}/{PATIENCE})")
        
        if epochs_no_improve >= PATIENCE:
            print(f"\n[EARLY STOPPING ATIVADO] O modelo não melhorou nas últimas {PATIENCE} épocas.")
            history['train_loss'] = history['train_loss'][:-PATIENCE]
            history['train_acc'] = history['train_acc'][:-PATIENCE]
            history['val_loss'] = history['val_loss'][:-PATIENCE]
            history['val_acc'] = history['val_acc'][:-PATIENCE]
            break

if os.path.exists(TEMP_MODEL_PATH):
    shutil.move(TEMP_MODEL_PATH, MODEL_SAVE_PATH)
    print(f"\n[SUCESSO] Melhor modelo salvo em: {MODEL_SAVE_PATH}")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))

Epoch [01/80] Train Loss: 0.1006 | Train Acc: 0.5105 | Val Loss: 0.0936 | Val Acc: 0.5245
  -> Melhoria detectada! Loss caiu para 0.0936. (Patience: 0/10)
Epoch [02/80] Train Loss: 0.0938 | Train Acc: 0.5647 | Val Loss: 0.0948 | Val Acc: 0.5315
  -> Sem melhoria. (Patience: 1/10)
Epoch [03/80] Train Loss: 0.0924 | Train Acc: 0.5747 | Val Loss: 0.0906 | Val Acc: 0.5734
  -> Melhoria detectada! Loss caiu para 0.0906. (Patience: 0/10)
Epoch [04/80] Train Loss: 0.0877 | Train Acc: 0.6008 | Val Loss: 0.0820 | Val Acc: 0.6713
  -> Melhoria detectada! Loss caiu para 0.0820. (Patience: 0/10)
Epoch [05/80] Train Loss: 0.0877 | Train Acc: 0.5998 | Val Loss: 0.0920 | Val Acc: 0.5524
  -> Sem melhoria. (Patience: 1/10)
Epoch [06/80] Train Loss: 0.0870 | Train Acc: 0.6189 | Val Loss: 0.1003 | Val Acc: 0.5734
  -> Sem melhoria. (Patience: 2/10)
Epoch [07/80] Train Loss: 0.0858 | Train Acc: 0.6028 | Val Loss: 0.0951 | Val Acc: 0.5594
  -> Sem melhoria. (Patience: 3/10)
Epoch [08/80] Train Loss: 0.085

### Gráficos

In [8]:
# GRÁFICOS DE TREINAMENTO (HISTÓRICO)
epochs = range(1, len(history['train_loss']) + 1)
train_loss = history['train_loss']
val_loss = history['val_loss']
val_acc = history['val_acc']
train_acc = history.get('train_acc', None) 

# 1. Accuracy vs. epoch (Treino vs. Validação)
plt.figure(figsize=(6, 5))
if train_acc is not None and len(train_acc) > 0:
    plt.plot(epochs, train_acc, color='tab:blue', label='Training accuracy', linewidth=1.5)
plt.plot(epochs, val_acc, color='tab:orange', label='Validation accuracy', linewidth=1.5)
plt.xlabel('Epoch', fontweight='bold', fontsize=12)
plt.ylabel('Accuracy', fontweight='bold', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.savefig(os.path.join(RUNS_DIR, "accuracy_vs_epoch.png"), dpi=300, bbox_inches='tight')
plt.close() 

# 2. Loss vs. epoch
plt.figure(figsize=(6, 5))
plt.plot(epochs, train_loss, color='tab:blue', label='Training loss', linewidth=1.5)
plt.plot(epochs, val_loss, color='tab:orange', label='Validation loss', linewidth=1.5)
plt.xlabel('Epoch', fontweight='bold', fontsize=12)
plt.ylabel('Loss', fontweight='bold', fontsize=12)
plt.legend(loc='upper right', fontsize=11)
plt.savefig(os.path.join(RUNS_DIR, "loss_vs_epoch.png"), dpi=300, bbox_inches='tight')
plt.close()

# INFERÊNCIA NO CONJUNTO DE TESTE
model.eval()
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        
        # Extrai a probabilidade usando softmax (necessário para as curvas ROC e PR)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

class_names = full_dataset.classes 
n_classes = len(class_names)

# 3. Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 16}, cbar_kws={'shrink': 0.8})

plt.xlabel('Predicted Labels', fontweight='bold', fontsize=12)
plt.ylabel('True Labels', fontweight='bold', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.savefig(os.path.join(RUNS_DIR, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.close()

# Binarização dos rótulos para curvas multiclasse (ROC e Precision-Recall)
y_test_bin = label_binarize(all_labels, classes=range(n_classes))
colors = sns.color_palette("husl", n_classes)

# 4. ROC Curve
plt.figure(figsize=(6, 5))
roc_auc_dict = {}

if n_classes == 2:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color='tab:orange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
else:
    for i, color in zip(range(n_classes), colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        roc_auc_dict[class_names[i]] = roc_auc
        plt.plot(fpr, tpr, color=color, lw=1.5, label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color='gray')
plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
plt.legend(loc="lower right", fontsize=10)
plt.savefig(os.path.join(RUNS_DIR, "roc_curve.png"), dpi=300, bbox_inches='tight')
plt.close()

# 5. Precision-Recall Curve (Ideal para avaliar classes desbalanceadas como 'miner')
plt.figure(figsize=(6, 5))
pr_auc_dict = {}

if n_classes > 2:
    for i, color in zip(range(n_classes), colors):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, i], all_probs[:, i])
        ap_score = average_precision_score(y_test_bin[:, i], all_probs[:, i])
        pr_auc_dict[class_names[i]] = ap_score
        plt.plot(recall, precision, color=color, lw=1.5, label=f'{class_names[i]} (AP = {ap_score:.4f})')

    plt.xlabel('Recall (Sensibilidade)', fontweight='bold', fontsize=12)
    plt.ylabel('Precision (Precisão)', fontweight='bold', fontsize=12)
    plt.legend(loc="lower left", fontsize=10)
    plt.savefig(os.path.join(RUNS_DIR, "pr_curve.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 6. RELATÓRIO DE MÉTRICAS REAIS NO CONJUNTO DE TESTE
bal_acc = balanced_accuracy_score(all_labels, all_preds)
report_str = classification_report(all_labels, all_preds, target_names=class_names, digits=4)

resumo_metricas = (
    "============================================================\n"
    "          AVALIAÇÃO REAL NO CONJUNTO DE TESTE               \n"
    "============================================================\n"
    f"Acurácia Global (Padrão):     {(all_preds == all_labels).mean():.4f}\n"
    f"Acurácia Balanceada (Real):   {bal_acc:.4f}\n\n"
    "--- Relatório Detalhado por Classe ---\n"
    f"{report_str}\n"
    "--- AUC-ROC vs. Average Precision (PR-AUC) por Classe ---\n"
)

for cls_name in class_names:
    r_auc = roc_auc_dict.get(cls_name, 0.0)
    p_auc = pr_auc_dict.get(cls_name, 0.0)
    resumo_metricas += f" - {cls_name:<10} | ROC-AUC: {r_auc:.4f} | PR-AUC (AP): {p_auc:.4f}\n"

resumo_metricas += "============================================================\n"

print(resumo_metricas)

# Salva o relatório em texto junto com os gráficos
report_path = os.path.join(RUNS_DIR, "relatorio_metricas.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(resumo_metricas)

print(f"[SUCESSO] Gráficos e relatório de métricas salvos em: {RUNS_DIR}")

          AVALIAÇÃO REAL NO CONJUNTO DE TESTE               
Acurácia Global (Padrão):     0.8776
Acurácia Balanceada (Real):   0.8022

--- Relatório Detalhado por Classe ---
                 precision    recall  f1-score   support

        healthy     0.9290    0.9863    0.9568       146
red_spider_mite     0.6333    0.6129    0.6230        31
           rust     0.8713    0.8073    0.8381       109

       accuracy                         0.8776       286
      macro avg     0.8112    0.8022    0.8060       286
   weighted avg     0.8750    0.8776    0.8754       286

--- AUC-ROC vs. Average Precision (PR-AUC) por Classe ---
 - healthy    | ROC-AUC: 0.9674 | PR-AUC (AP): 0.9551
 - red_spider_mite | ROC-AUC: 0.9399 | PR-AUC (AP): 0.6657
 - rust       | ROC-AUC: 0.9377 | PR-AUC (AP): 0.9247

[SUCESSO] Gráficos e relatório de métricas salvos em: ../runs/FusNet_run
